# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [2]:
import pandas as pd

urls = [
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv",
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv",
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"
]

In [11]:
raw_dataframes = []

for url in urls:
    df = pd.read_csv(url)
    raw_dataframes.append(df)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print()

Shape: (4008, 11)
Columns: ['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Vehicle Class', 'Total Claim Amount']

Shape: (996, 11)
Columns: ['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Total Claim Amount', 'Policy Type', 'Vehicle Class']

Shape: (7070, 11)
Columns: ['Customer', 'State', 'Customer Lifetime Value', 'Education', 'Gender', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Total Claim Amount', 'Vehicle Class']



In [12]:
dataframes = []

for df in raw_dataframes:

    # Remove completely empty rows
    df = df.dropna(how="all")

    # Standardise column names
    df.columns = (
        df.columns
        .str.lower()
        .str.replace(" ", "_")
    )

    # Make the state column consistent
    df = df.rename(columns={"st": "state"})

    # Convert CLV values to numeric
    clv = df["customer_lifetime_value"].astype("string")
    has_percentage = clv.str.endswith("%", na=False)

    df["customer_lifetime_value"] = pd.to_numeric(
        clv.str.rstrip("%"),
        errors="coerce"
    )

    df.loc[has_percentage, "customer_lifetime_value"] = (
        df.loc[has_percentage, "customer_lifetime_value"] / 100
    )

    # Extract the complaint number from values such as "1/2/00"
    complaints = df["number_of_open_complaints"].astype("string")
    complaint_number = complaints.str.split("/").str[1]

    df["number_of_open_complaints"] = pd.to_numeric(
        complaint_number.fillna(complaints),
        errors="coerce"
    )

    dataframes.append(df)

In [13]:
# Combine the datasets
customer_data = pd.concat(dataframes, ignore_index=True)

In [15]:
# Remove exact duplicate rows
customer_data = customer_data.drop_duplicates().reset_index(drop=True)

customer_data.info()
customer_data.isna().sum()
customer_data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9134 entries, 0 to 9133
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer                   9134 non-null   object 
 1   state                      9134 non-null   object 
 2   gender                     9012 non-null   object 
 3   education                  9134 non-null   object 
 4   customer_lifetime_value    9127 non-null   Float64
 5   income                     9134 non-null   float64
 6   monthly_premium_auto       9134 non-null   float64
 7   number_of_open_complaints  9134 non-null   int64  
 8   policy_type                9134 non-null   object 
 9   vehicle_class              9134 non-null   object 
 10  total_claim_amount         9134 non-null   float64
dtypes: Float64(1), float64(3), int64(1), object(6)
memory usage: 794.0+ KB


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,NaN,Master,<NA>,0.0,1000.0,0,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,6979.5359,0.0,94.0,0,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,12887.4317,48767.0,108.0,0,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,7645.8618,0.0,106.0,0,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,5363.0765,36357.0,68.0,0,Personal Auto,Four-Door Car,17.269323


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [7]:
marketing = pd.read_csv(
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
)

marketing = marketing.drop(columns=["unnamed:_0"])

In [8]:
marketing["effective_to_date"] = pd.to_datetime(
    marketing["effective_to_date"]
)

marketing.head()

,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,48029,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,A,2
1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,0,...,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,A,1
2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,22139,...,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A,2
3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,49078,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,2011-01-17,Medical Leave,F,23675,...,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,A,1


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

In [10]:
revenue_by_channel = marketing.pivot_table(
    index="sales_channel",
    values="customer_lifetime_value",
    aggfunc="sum"
).round(2)

revenue_by_channel = revenue_by_channel.sort_values(
    by="customer_lifetime_value",
    ascending=False
)

revenue_by_channel

,customer_lifetime_value
sales_channel,
Agent,33057887.85
Branch,24359201.21
Call Center,17364288.37
Web,12697632.90


2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [9]:
average_clv = marketing.pivot_table(
    index="gender",
    columns="education",
    values="customer_lifetime_value",
    aggfunc="mean"
).round(2)

average_clv

education,Bachelor,College,Doctor,High School or Below,Master
gender,,,,,
F,7874.27,7748.82,7328.51,8675.22,8157.05
M,7703.60,8052.46,7415.33,8149.69,8168.83


## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
# Your code goes here